<a href="https://colab.research.google.com/github/mariam9809m1-gif/MERN-Task-Architect/blob/main/Real_Estate_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import gradio as gr

# 1. Synthetic Data Generation
np.random.seed(42)
num_samples = 1000

# Generate realistic feature data
shared_rooms = np.random.randint(1, 6, num_samples) # Number of rooms
square_footage = np.random.randint(500, 5000, num_samples)
neighborhood_quality = np.random.uniform(1.0, 5.0, num_samples) # e.g., 1=bad, 5=excellent
year_built = np.random.randint(1950, 2023, num_samples)

# Generate price based on features, adding some noise
# Base price tends to increase with square footage, more rooms, better neighborhood, and newer year built
price = (50 * square_footage +
         20000 * shared_rooms +
         50000 * neighborhood_quality +
         1000 * (2023 - year_built) * -1 +
         np.random.normal(0, 50000, num_samples))

# Ensure prices are positive
price = np.maximum(100000, price)

data = pd.DataFrame({
    'SharedRooms': shared_rooms,
    'SquareFootage': square_footage,
    'Neighborhood_Quality': neighborhood_quality,
    'YearBuilt': year_built,
    'Price': price
})

print("Generated synthetic data head:")
display(data.head())

# 2. Machine Learning Core

X = data[['SharedRooms', 'SquareFootage', 'Neighborhood_Quality', 'YearBuilt']]
y = data['Price']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train a RandomForestRegressor model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

print(f"\nModel R^2 score on test set: {model.score(X_test_scaled, y_test):.2f}")

# 3. Interactive Dashboard Layer with Gradio
def predict_price(shared_rooms_input, square_footage_input, neighborhood_quality_input, year_built_input):
    # Create a DataFrame from the inputs
    input_data = pd.DataFrame([[shared_rooms_input, square_footage_input, neighborhood_quality_input, year_built_input]],
                              columns=['SharedRooms', 'SquareFootage', 'Neighborhood_Quality', 'YearBuilt'])

    # Scale the input data using the trained scaler
    input_scaled = scaler.transform(input_data)

    # Make a prediction
    predicted_price = model.predict(input_scaled)[0]

    return f"${predicted_price:,.2f}"

# Gradio Interface
iface = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Slider(minimum=1, maximum=10, step=1, value=3, label="Number of Shared Rooms"),
        gr.Slider(minimum=200, maximum=10000, step=50, value=2000, label="Square Footage (sqft)"),
        gr.Slider(minimum=1.0, maximum=5.0, step=0.1, value=3.0, label="Neighborhood Quality (1-5)"),
        gr.Slider(minimum=1900, maximum=2023, step=1, value=2000, label="Year Built")
    ],
    outputs="text",
    title="Real Estate Price Prediction",
    description="Enter house details to get an estimated price. Model uses RandomForestRegressor."
)

# 4. Deployment: Launch Gradio interface with a public shareable link
print("\nLaunching Gradio interface...")
iface.launch(share=True)


Generated synthetic data head:


,SharedRooms,SquareFootage,Neighborhood_Quality,YearBuilt,Price
0,4,4031,2.290112,1962,344247.321385
1,5,4951,1.091575,2006,519780.448194
2,3,3435,1.615165,1998,304998.231162
3,5,4376,1.894458,2003,343320.157057
4,5,626,4.829292,1959,303991.365224



Model R^2 score on test set: 0.76

Launching Gradio interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f917599eca320efa14.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
